## Import libraries

In [1]:
import numpy as np
import pandas as pd
from glob import glob
import os, shutil
from obspy import read, UTCDateTime as utc
from tqdm import tqdm

## Load selected station list with sensor type

In [3]:
used_station_list=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/station_table/used_stations_100km.csv", sep=",")
used_station_list.head(5)

,Network,Code,Latitude,Longitude,Elevation,Sensor
0,KS,BAU,35.1027,127.5968,562.0,EL
1,KS,BLGA,36.1392,127.3032,105.0,HG
2,KS,BLLA,34.8646,127.4638,45.0,HG
3,KS,BUYB,36.2726,126.9206,59.0,HH
4,KS,CGPA,34.8768,126.9710,144.0,HG


## Move SAC files corresponding to selected stations to separate directory for analysis

In [4]:
destination_dir="/home/msseo/works/72.2025_Imsil_swarm/waveforms_100km/"

eventdir_list=sorted(glob("/home/msseo/works/72.2025_Imsil_swarm/kma_waveforms/20*"))
for eventdir in eventdir_list:
    datetime=eventdir.split("/")[-1]
    new_eventdir=destination_dir+datetime
    os.makedirs(new_eventdir, exist_ok=True)
    
    # HH
    HH_list=glob(eventdir+"/*.v/SAC/HH/*HH*SAC")
    for sacfile in HH_list:
        code=sacfile.split("/")[-1].split(".")[1]
        if code in np.array(used_station_list[used_station_list["Sensor"]=="HH"]["Code"]):
            sacfile_name=sacfile.split("/")[-1]
            new_sacfile=new_eventdir+"/"+sacfile_name
            shutil.copyfile(sacfile, new_sacfile)
        
    # HG
    HG_list=glob(eventdir+"/*.a/SAC/HG/*HG*SAC")
    for sacfile in HG_list:
        code=sacfile.split("/")[-1].split(".")[1]
        if code in np.array(used_station_list[used_station_list["Sensor"]=="HG"]["Code"]):
            sacfile_name=sacfile.split("/")[-1]
            new_sacfile=new_eventdir+"/"+sacfile_name
            shutil.copyfile(sacfile, new_sacfile)
    
    # EL
    EL_list=glob(eventdir+"/*.v/SAC/EL/*EL*SAC")
    for sacfile in EL_list:
        code=sacfile.split("/")[-1].split(".")[1]
        if code in np.array(used_station_list[used_station_list["Sensor"]=="EL"]["Code"]):
            sacfile_name=sacfile.split("/")[-1]
            new_sacfile=new_eventdir+"/"+sacfile_name
            shutil.copyfile(sacfile, new_sacfile)

## Load KMA event catalog

In [5]:
event_catalog=pd.read_csv("/home/msseo/works/72.2025_Imsil_swarm/event_catalog/Imsil_catalog.csv", sep=",")
event_catalog

,Year,Month,Day,Hour,Minute,Second,Latitude,Longitude,Depth,Magnitude
0,2025,11,3,5,5,39,35.55,127.29,13,1.4
1,2025,11,3,5,6,25,35.55,127.29,12,1.4
2,2025,11,3,5,10,44,35.55,127.30,13,1.3
3,2025,11,3,5,23,39,35.56,127.30,13,1.5
4,2025,11,3,5,24,12,35.55,127.29,14,1.5
5,2025,11,3,5,29,39,35.55,127.30,12,1.0
6,2025,11,3,5,32,36,35.55,127.29,13,1.7
7,2025,11,3,5,36,49,35.55,127.29,13,1.3
8,2025,11,3,5,38,35,35.55,127.30,14,1.9
9,2025,11,3,5,40,2,35.55,127.30,13,1.3


## Input origin time, station, event coordinates in SAC header

In [6]:
eventdir_list=sorted(glob("/home/msseo/works/72.2025_Imsil_swarm/waveforms_100km/20*"))
for (i, eventdir) in tqdm(enumerate(eventdir_list)):
    sacfile_list=glob(eventdir+"/*.SAC")
    year, month, day=event_catalog["Year"][i], event_catalog["Month"][i], event_catalog["Day"][i]
    hour, minute, second=event_catalog["Hour"][i], event_catalog["Minute"][i], event_catalog["Second"][i]
    origin_time=utc(year, month, day, hour, minute, second)-9*3600
    evla, evlo=event_catalog["Latitude"][i], event_catalog["Longitude"][i]
    
    for sacfile in sacfile_list:
        code=sacfile.split("/")[-1].split(".")[1]
        stla=used_station_list[used_station_list["Code"]==code].reset_index(drop=True)["Latitude"][0]
        stlo=used_station_list[used_station_list["Code"]==code].reset_index(drop=True)["Longitude"][0]
        tr=read(sacfile)[0]
        tr.stats.sac.evla, tr.stats.sac.evlo=evla, evlo
        tr.stats.sac.stla, tr.stats.sac.stlo=stla ,stlo
        tr.stats.sac.nzyear=origin_time.year
        tr.stats.sac.nzjday=origin_time.julday
        tr.stats.sac.nzhour=origin_time.hour
        tr.stats.sac.nzmin=origin_time.minute
        tr.stats.sac.nzsec=origin_time.second
        
        tr.write(sacfile, format="SAC")

24it [00:21,  1.09it/s]
